In [85]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"C:\Users\dbila\Desktop\airtel-telecom-analytics\data\raw\airtel.db")
def q(sql):
    return pd.read_sql(sql, conn)


In [86]:
q("""
SELECT COUNT(*)
FROM customer_features;
""")

,COUNT(*)
0,22000


In [87]:
q("""
SELECT ROUND(AVG(actual_arpu_6m), 2)
FROM customer_features;
""")

,"ROUND(AVG(actual_arpu_6m), 2)"
0,344.46


In [88]:
q("""
SELECT SUM(complaint_count = 0)
FROM customer_features;
""")

,SUM(complaint_count = 0)
0,6631


In [89]:
q("""
SELECT ab_group, COUNT(*)
FROM customer_features
GROUP BY ab_group;
""")

,ab_group,COUNT(*)
0,A,10996
1,B,11004


In [90]:
q("""
SELECT connection_type,
       COUNT(*) AS subs,
       ROUND(100.0*COUNT(*)/22000, 2) AS pct
FROM customer_features
GROUP BY connection_type
ORDER BY subs DESC;
""")

,connection_type,subs,pct
0,Prepaid,17149,77.95
1,Postpaid,4851,22.05


# Airtel SQL Data Analysis — Areas A to F


# AREA A — THE SUBSCRIBER BASE

### A1 — Subscribers by connection 

 How many subscribers are prepaid and how many are postpaid? Calculate each group's percentage of the total base.

In [91]:
q("""
SELECT connection_type,                         -- prepaid or postpaid
       COUNT(*) AS subs,                         -- subscribers in each group
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 2) AS pct -- share of total base
FROM customer_features                          -- one row per subscriber
GROUP BY connection_type                        -- separate connection types
ORDER BY subs DESC;                             -- largest group first
""")

,connection_type,subs,pct
0,Prepaid,17149,77.95
1,Postpaid,4851,22.05


### A2 — Five most popular plans
Which five plans have the most subscribers? Show plan name, plan type, price and subscriber count.

In [92]:
q("""
SELECT plan_name, plan_type, price_inr,              -- plan details
       COUNT(*) AS subs                              -- subscribers on each plan
FROM customer_features                              -- one row per subscriber
GROUP BY plan_name, plan_type, price_inr             -- one row per plan
ORDER BY subs DESC                                   -- most popular first
LIMIT 5;                                             -- return five plans
""")

,plan_name,plan_type,price_inr,subs
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


### A3 — Five circles with most subscribers 
Which five circles have the most subscribers? Include region and subscriber count.

In [93]:
q("""
SELECT circle_name, region,                          -- circle and region
       COUNT(*) AS subs                              -- subscribers in circle
FROM customer_features
GROUP BY circle_name, region                         -- one row per circle/region
ORDER BY subs DESC
LIMIT 5;
""")

,circle_name,region,subs
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


### A4 — Subscriber count by age band
How many subscribers are in each age band? Show count and percentage, ordered by age band.

In [94]:
q("""
SELECT age_band,                                 -- age category
       COUNT(*) AS subs,                         -- subscribers in band
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct -- percentage of base
FROM customer_features
GROUP BY age_band
ORDER BY age_band;                               -- natural band order
""")

,age_band,subs,pct
0,18-24,5268,23.9
1,25-34,7467,33.9
2,35-44,4824,21.9
3,45-54,2895,13.2
4,55+,1546,7.0


### A5 — Acquisition channel
How many subscribers came from each acquisition channel? Show count and percentage, highest first.

In [95]:
q("""
SELECT acquisition_channel,                     -- source of acquisition
       COUNT(*) AS subs,                         -- subscribers from channel
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct -- share of total base
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subs DESC;
""")

,acquisition_channel,subs,pct
0,Retailer,9287,42.2
1,Airtel App,6513,29.6
2,Website,3081,14.0
3,Call Center,1770,8.0
4,Doorstep,1349,6.1


### A6 — Device type and operating system
How many subscribers use each device type and operating system combination? Count each combination.

In [96]:
q("""
SELECT device_type, os,                          -- device and OS combination
       COUNT(*) AS subs                          -- subscribers in combination
FROM customer_features
GROUP BY device_type, os                         -- one row per combination
ORDER BY subs DESC;
""")

,device_type,os,subs
0,Smartphone,Android,15613
1,Smartphone,iOS,3358
2,Feature Phone,Other,3029


### A7 — Subscriber tenure distribution
 How many subscribers have 0, 1, 2 and 3 completed years of tenure? Show count and percentage.

In [97]:
q("""
SELECT tenure_years,                             -- completed years with company
       COUNT(*) AS subs,                         -- subscribers at each tenure
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct -- percentage of base
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,pct
0,0,4610,21.0
1,1,6961,31.6
2,2,7055,32.1
3,3,3374,15.3


# AREA B — REVENUE AND ARPU

### B1 — Average stated ARPU
 Calculate the average stated_arpu across subscribers. Also show the number of subscribers included.

In [98]:
q("""
SELECT ROUND(AVG(stated_arpu), 2) AS avg_arpu,     -- average stated plan price
       COUNT(*) AS subs                            -- number of subscribers included
FROM customer_features;
""")

,avg_arpu,subs
0,421.65,22000


### B2 — Average actual ARPU and monthly total
Calculate the average actual_arpu_6m and the total monthly revenue represented by this measure.

In [99]:
q("""
SELECT ROUND(AVG(actual_arpu_6m), 2) AS avg_actual, -- average actual ARPU
       ROUND(SUM(actual_arpu_6m), 0) AS monthly_total -- total monthly revenue represented
FROM customer_features;
""")

,avg_actual,monthly_total
0,344.46,7578210.0


### B3 — Difference between stated and actual ARPU
Compare stated_arpu with actual_arpu_6m. Calculate the average gap and percentage difference. State the denominator used.

In [100]:
q("""
SELECT ROUND(AVG(stated_arpu), 2) AS stated,       -- average stated amount
       ROUND(AVG(actual_arpu_6m), 2) AS actual,    -- average received amount
       ROUND(AVG(arpu_gap), 2) AS gap,             -- average difference
       ROUND(100.0 * (SUM(stated_arpu) - SUM(actual_arpu_6m))
             / NULLIF(SUM(actual_arpu_6m), 0), 2) AS over_pct -- gap as % of actual revenue
FROM customer_features;
""")

,stated,actual,gap,over_pct
0,421.65,344.46,77.18,22.41


### B4 — Stated and actual ARPU by connection type
 Compare stated and actual ARPU for prepaid and postpaid subscribers separately.

In [101]:
q("""
SELECT connection_type,                            -- prepaid or postpaid
       COUNT(*) AS subs,                            -- group size
       ROUND(AVG(stated_arpu), 2) AS stated,        -- plan-price ARPU
       ROUND(AVG(actual_arpu_6m), 2) AS actual      -- actual ARPU measure
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subs,stated,actual
0,Postpaid,4851,959.65,970.42
1,Prepaid,17149,269.46,167.40


### B5 — Five plans with highest actual monthly revenue
 Find the five plans with the highest actual monthly revenue. Show subscriber count, total revenue and revenue per subscriber.

In [102]:
q("""
SELECT plan_name,                                   -- plan name
       COUNT(*) AS subs,                            -- subscribers on plan
       ROUND(SUM(actual_arpu_6m), 0) AS rev,        -- total monthly revenue
       ROUND(AVG(actual_arpu_6m), 2) AS per_sub     -- revenue per subscriber
FROM customer_features
GROUP BY plan_name
ORDER BY rev DESC
LIMIT 5;
""")

,plan_name,subs,rev,per_sub
0,Postpaid Infinity 1499,1431,2171756.0,1517.65
1,Postpaid Family 999,1239,1249246.0,1008.27
2,Postpaid 749,946,718959.0,760.00
3,Prepaid Entertainment 399,2142,671850.0,313.66
4,Prepaid Data 349,1884,520066.0,276.04


### B6 — Five circles with highest actual monthly revenue
Find the five circles with the highest actual monthly revenue. Show subscriber count and total revenue.

In [103]:
q("""
SELECT circle_name,                                 -- circle name
       COUNT(*) AS subs,                            -- subscribers in circle
       ROUND(SUM(actual_arpu_6m), 0) AS rev         -- total actual revenue
FROM customer_features
GROUP BY circle_name
ORDER BY rev DESC
LIMIT 5;
""")

,circle_name,subs,rev
0,Mumbai,1395,473288.0
1,Karnataka,1361,468515.0
2,Andhra Pradesh,1412,463370.0
3,North East,1318,460998.0
4,Maharashtra,1428,450032.0


### B7 — Payments and amount received January–June 2025
 Calculate the number of payment transactions and total amount received from 1 January to 30 June 2025 using recharges.

In [104]:
q("""
SELECT COUNT(*) AS payments,                     -- number of transactions
       ROUND(SUM(amount_inr) / 10000000.0, 2) AS crore_inr -- amount in crore INR
FROM recharges                                   -- one row per payment
WHERE recharge_date >= '2025-01-01'              -- inclusive start date
  AND recharge_date < '2025-07-01';              -- exclusive end date includes all of June
""")

,payments,crore_inr
0,83693,4.55


### B8 — Actual ARPU by tenure
 Compare average actual_arpu_6m for subscribers with 0, 1, 2 and 3 years of tenure.

In [105]:
q("""
SELECT tenure_years,                                -- completed years
       COUNT(*) AS subs,                            -- subscribers in group
       ROUND(AVG(actual_arpu_6m), 2) AS arpu        -- average actual ARPU
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,arpu
0,0,4610,329.10
1,1,6961,342.72
2,2,7055,350.13
3,3,3374,357.22


### B9 — Subscribers with no recharge and churn
Count subscribers with recharge_count = 0, calculate the churn percentage for that group, and inspect stated_arpu carefully. Explain why it may be misleading.

In [106]:
q("""
SELECT SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END) AS paid_nothing, -- zero-recharge subscribers
       ROUND(100.0 * AVG(CASE WHEN recharge_count = 0 THEN is_churned END), 2) AS their_churn, -- churn within group
       ROUND(AVG(CASE WHEN recharge_count = 0 THEN stated_arpu END), 2) AS zero_recharge_stated_arpu -- stated ARPU in that group
FROM customer_features;
""")

,paid_nothing,their_churn,zero_recharge_stated_arpu
0,3711,44.3,186.2


# AREA C — RECHARGE BEHAVIOUR

### C1 — Payment count and distinct subscribers by transaction type
For each txn_type, count payment transactions and distinct subscribers.

In [107]:
q("""
SELECT txn_type,                                   -- transaction type
       COUNT(*) AS payments,                       -- number of payment rows
       COUNT(DISTINCT customer_id) AS subs         -- unique subscribers
FROM recharges
GROUP BY txn_type;
""")

,txn_type,payments,subs
0,Bill Payment,108419,4851
1,Recharge,299670,17149


### C2 — Overall average payment
Calculate the average amount_inr across all rows in recharges. Record the result and explain what population it averages.

In [108]:
q("""
SELECT ROUND(AVG(amount_inr), 2) AS avg_payment, -- average across all payment rows
       COUNT(*) AS payment_rows                    -- denominator: payment transactions
FROM recharges;
""")

,avg_payment,payment_rows
0,532.59,408089


### C3 — Average, minimum and maximum by transaction type
 Calculate average, minimum and maximum payment amount separately for each txn_type.

In [109]:
q("""
SELECT txn_type,                                   -- payment category
       ROUND(AVG(amount_inr), 2) AS avg_amt,       -- average payment
       MIN(amount_inr) AS mn,                     -- smallest payment
       MAX(amount_inr) AS mx                      -- largest payment
FROM recharges
GROUP BY txn_type;
""")

,txn_type,avg_amt,mn,mx
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


### C4 — Payment frequency and total by connection type
For prepaid and postpaid subscribers, calculate average recharge_count and average recharge_total_6m.

In [110]:
q("""
SELECT connection_type,                           -- prepaid or postpaid
       ROUND(AVG(recharge_count), 2) AS avg_payments, -- average number of payments
       ROUND(AVG(recharge_total_6m), 0) AS avg_total_6m -- average six-month total
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,avg_payments,avg_total_6m
0,Postpaid,5.39,5823.0
1,Prepaid,3.36,1004.0


### C5 — Payment channels by transaction count
Find the payment channels with the highest number of transactions. Show count and percentage of all payments.

In [111]:
q("""
SELECT channel,                                   -- payment channel
       COUNT(*) AS payments,                      -- transactions on channel
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM recharges), 1) AS pct -- share of all payments
FROM recharges
GROUP BY channel
ORDER BY payments DESC;
""")

,channel,payments,pct
0,Website,111163,27.2
1,Airtel App,111083,27.2
2,USSD,75257,18.4
3,Retailer,74406,18.2
4,AutoPay,36180,8.9


### C6 — Channel share within each transaction type
Compare payment-channel usage for prepaid and postpaid transactions. Calculate each channel's percentage within its own txn_type group.

In [112]:
q("""
SELECT txn_type, channel,                         -- transaction type and channel
       COUNT(*) AS n,                             -- number of transactions
       ROUND(100.0 * COUNT(*) /
             SUM(COUNT(*)) OVER (PARTITION BY txn_type), 1) AS pct -- share within txn_type
FROM recharges
GROUP BY txn_type, channel
ORDER BY txn_type, n DESC;
""")

,txn_type,channel,n,pct
0,Bill Payment,Airtel App,36186,33.4
1,Bill Payment,AutoPay,36180,33.4
2,Bill Payment,Website,36053,33.3
3,Recharge,USSD,75257,25.1
4,Recharge,Website,75110,25.1
5,Recharge,Airtel App,74897,25.0
6,Recharge,Retailer,74406,24.8


### C7 — Churn by subscriber recharge count
Group subscribers by recharge_count and calculate subscriber count and churn percentage for each count. Restrict the displayed range to counts of eight or fewer and comment on small groups.

In [113]:
q("""
SELECT recharge_count,                            -- number of payments
       COUNT(*) AS subs,                          -- subscribers at this count
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM customer_features
WHERE recharge_count <= 8                         -- requested range
GROUP BY recharge_count
ORDER BY recharge_count;
""")

,recharge_count,subs,churn
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63


# AREA D — DAILY USAGE OVER TIME

### D1 — Monthly network traffic and active subscribers
For each month, calculate total data traffic in TB and the number of distinct active subscribers.

In [114]:
q("""
SELECT month,                                    -- month bucket
       COUNT(DISTINCT customer_id) AS active_subs, -- distinct active subscribers
       ROUND(SUM(total_mb) / 1024.0 / 1024.0, 1) AS total_tb -- MB to TB
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,active_subs,total_tb
0,2025-01,21302,338.0
1,2025-02,21162,307.3
2,2025-03,20366,332.5
3,2025-04,19320,310.3
4,2025-05,18356,363.1
5,2025-06,17326,338.9


### D2 — Average daily data across usage rows
Calculate average data_mb across all rows in usage_daily and show the number of rows used. Explain what the average is divided by.

In [115]:
q("""
SELECT ROUND(AVG(data_mb), 2) AS avg_mb,          -- average per subscriber-day row
       COUNT(*) AS rows_                          -- denominator: subscriber-day rows
FROM usage_daily;
""")

,avg_mb,rows_
0,1572.87,1326739


### D3 — Average per active day and per observed day
From customer_features, calculate average mb_per_active_day and mb_per_observed_day. Compare the measures and explain their denominators.

In [116]:
q("""
SELECT ROUND(AVG(mb_per_active_day), 2) AS per_active_day, -- per day used
       ROUND(AVG(mb_per_observed_day), 2) AS per_observed_day, -- per day observed
       COUNT(*) AS subs                           -- subscriber rows included
FROM customer_features;
""")

,per_active_day,per_observed_day,subs
0,1404.44,524.05,22000


### D4 — Daily data usage by weekday
Compare average daily data_mb across the seven weekdays using strftime('%w', usage_date). Show the number of rows in each group.

In [117]:
q("""
SELECT strftime('%w', usage_date) AS dow,          -- weekday: 0 Sunday to 6 Saturday
       COUNT(*) AS rows_,                          -- subscriber-day rows
       ROUND(AVG(data_mb), 1) AS avg_mb            -- average data on weekday
FROM usage_daily
GROUP BY dow
ORDER BY dow;
""")

,dow,rows_,avg_mb
0,0,190368,1578.1
1,1,189914,1572.9
2,2,183823,1569.5
3,3,191088,1564.5
4,4,190764,1580.0
5,5,190542,1569.5
6,6,190240,1575.5


### D5 — Active subscribers by month
Count distinct active subscribers in each month and describe the trend.

In [118]:
q("""
SELECT month,                                    -- month bucket
       COUNT(DISTINCT customer_id) AS active_subs -- unique active subscribers
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,active_subs
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


### D6 — Average voice minutes and SMS by month
Calculate average voice_minutes and sms_count for each month from January to June 2025.

In [119]:
q("""
SELECT strftime('%Y-%m', usage_date) AS m,       -- extract year and month
       ROUND(AVG(voice_minutes), 1) AS avg_voice, -- average voice minutes
       ROUND(AVG(sms_count), 3) AS avg_sms        -- average SMS count
FROM usage_daily
WHERE usage_date >= '2025-01-01' AND usage_date < '2025-07-01' -- Jan–Jun 2025
GROUP BY m
ORDER BY m;
""")

,m,avg_voice,avg_sms
0,2025-01,55.6,0.745
1,2025-02,55.8,0.746
2,2025-03,56.1,0.750
3,2025-04,56.5,0.760
4,2025-05,56.7,0.764
5,2025-06,57.1,0.772


### D7 — Monthly data per subscriber and average active days
For each month, calculate average data use per subscriber in GB and average usage_days.

In [120]:
q("""
SELECT month,                                    -- month bucket
       ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb_per_sub, -- MB to GB
       ROUND(AVG(usage_days), 1) AS avg_days    -- average active days
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,avg_gb_per_sub,avg_days
0,2025-01,16.25,11.4
1,2025-02,14.87,10.3
2,2025-03,16.72,11.5
3,2025-04,16.44,11.2
4,2025-05,20.26,11.7
5,2025-06,20.03,11.5


### D8 — Monthly data per active day for never-churned subscribers
Restrict the analysis to subscribers who never churned. Join v_usage_month to customer_features and calculate their average data use per active day for each month.

In [121]:
q("""
SELECT m.month,                                  -- month bucket
       ROUND(AVG(m.total_mb * 1.0 / NULLIF(m.usage_days, 0)), 1) AS mb_per_day -- avoid zero division
FROM v_usage_month AS m                          -- one row per subscriber-month
JOIN customer_features AS c                      -- attach churn status
  ON c.customer_id = m.customer_id               -- match subscriber
WHERE c.is_churned = 0                           -- never-churned subscribers
GROUP BY m.month
ORDER BY m.month;
""")

,month,mb_per_day
0,2025-01,1410.6
1,2025-02,1422.1
2,2025-03,1418.1
3,2025-04,1411.8
4,2025-05,1659.5
5,2025-06,1662.2


# AREA E — USAGE INTENSITY AND DATA ALLOWANCE

### E1 — Four quartiles of daily data use
Exclude subscribers with no usage. Divide subscribers into four equal groups using NTILE(4) ordered by mb_per_active_day. Show group size and minimum and maximum daily use.

In [122]:
q("""
WITH n AS (
    SELECT mb_per_active_day AS m,                -- daily data measure
           NTILE(4) OVER (ORDER BY mb_per_active_day) AS q -- four equal groups
    FROM customer_features
    WHERE usage_days > 0                          -- exclude no-usage subscribers
)
SELECT q, COUNT(*) AS subs,                       -- quartile and group size
       ROUND(MIN(m), 0) AS mn,                    -- minimum daily use
       ROUND(MAX(m), 0) AS mx                     -- maximum daily use
FROM n
GROUP BY q
ORDER BY q;
""")

,q,subs,mn,mx
0,1,5499,25.0,657.0
1,2,5499,657.0,1093.0
2,3,5499,1093.0,1572.0
3,4,5499,1572.0,9720.0


### E2 — Churn by daily data-use quartile
Using the same four daily-data groups, calculate churn percentage in each group.

In [123]:
q("""
WITH n AS (
    SELECT is_churned,                            -- carry churn flag
           NTILE(4) OVER (ORDER BY mb_per_active_day) AS q -- daily data quartile
    FROM customer_features
    WHERE usage_days > 0
)
SELECT q, COUNT(*) AS subs,                       -- quartile size
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM n
GROUP BY q
ORDER BY q;
""")

,q,subs,churn
0,1,5499,30.28
1,2,5499,28.11
2,3,5499,22.51
3,4,5499,14.09


### E3 — Voice and data use by connection type and device
 Compare voice minutes per active day and data MB per active day for each connection_type and device_type combination.

In [124]:
q("""
SELECT connection_type, device_type,             -- subscriber segment
       COUNT(*) AS subs,                          -- subscribers in segment
       ROUND(AVG(total_voice_min * 1.0 /
                 NULLIF(usage_days, 0)), 1) AS voice_per_day, -- voice per active day
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day -- data per active day
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type, device_type
ORDER BY voice_per_day DESC;
""")

,connection_type,device_type,subs,voice_per_day,mb_per_day
0,Postpaid,Feature Phone,690,48.6,531.0
1,Postpaid,Smartphone,4161,48.0,3353.0
2,Prepaid,Feature Phone,2338,47.9,166.0
3,Prepaid,Smartphone,14807,47.9,1093.0


### E4 — Monthly allowance compared with actual monthly use by plan
For each plan, compare monthly data allowance (monthly_gb) with average actual monthly data use in GB.

In [125]:
q("""
SELECT plan_name, monthly_gb,                    -- plan and monthly allowance
       ROUND(AVG(total_data_mb) / 1024.0 / 6, 2) AS actual_gb_per_month -- six-month data to monthly GB
FROM customer_features
WHERE usage_days > 0
GROUP BY plan_name, monthly_gb
ORDER BY monthly_gb;
""")

,plan_name,monthly_gb,actual_gb_per_month
0,Prepaid Lite 99,15.0,2.88
1,Prepaid Smart 155,30.0,5.72
2,Prepaid Value 209,30.0,5.89
3,Postpaid 399,40.0,7.86
4,Prepaid Max 479,45.0,8.84
5,Prepaid Popular 299,45.0,8.93
6,Prepaid Annual 1799,60.0,11.87
7,Prepaid Quarterly 719,60.0,11.78
8,Postpaid 499,75.0,15.47
9,Prepaid Data 349,75.0,15.13


### E5 — Overall allowance usage and subscribers below half allowance
 Calculate average monthly allowance, average monthly usage, average percentage of allowance used, and number of subscribers using less than half their allowance.

In [126]:
q("""
SELECT ROUND(AVG(monthly_gb), 1) AS allowance_gb, -- average monthly allowance
       ROUND(AVG(total_data_mb / 1024.0 / 6), 2) AS used_gb, -- average monthly use
       ROUND(100.0 * AVG((total_data_mb / 1024.0 / 6) /
                         NULLIF(monthly_gb, 0)), 1) AS pct_of_allowance, -- average subscriber-level share
       SUM(CASE WHEN (total_data_mb / 1024.0 / 6) < monthly_gb * 0.5 THEN 1 ELSE 0 END) AS under_half
FROM customer_features
WHERE usage_days > 0;
""")

,allowance_gb,used_gb,pct_of_allowance,under_half
0,76.8,15.44,19.8,20560


### E6 — Spend, data use and churn by monetary score
 Group subscribers by m_score and compare six-month spend, data per active day and churn.

In [127]:

q("""
WITH scored AS (
    SELECT
        customer_id,
        NTILE(5) OVER (
            ORDER BY recharge_total_6m
        ) AS m_score,
        recharge_total_6m,
        mb_per_active_day,
        is_churned
    FROM customer_features
)
SELECT
    m_score,
    COUNT(*) AS subs,
    ROUND(AVG(recharge_total_6m), 0) AS spend_6m,
    ROUND(AVG(mb_per_active_day), 0) AS mb_per_day,
    ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM scored
GROUP BY m_score
ORDER BY m_score;
""")


,m_score,subs,spend_6m,mb_per_day,churn
0,1,4400,50.0,994.0,53.02
1,2,4400,792.0,902.0,31.11
2,3,4400,1245.0,712.0,13.80
3,4,4400,1986.0,1327.0,11.36
4,5,4400,6261.0,3087.0,9.48


### E7 — Churn by usage-days quartile
Divide subscribers into four equal groups by usage_days. Show each group's usage-day range and churn percentage.

In [128]:
q("""
WITH n AS (
    SELECT usage_days, is_churned,                 -- carry days and churn flag
           NTILE(4) OVER (ORDER BY usage_days) AS q -- four equal groups
    FROM customer_features
)
SELECT q, COUNT(*) AS subs,                       -- group size
       MIN(usage_days) AS mn,                      -- minimum active days
       MAX(usage_days) AS mx,                      -- maximum active days
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM n
GROUP BY q
ORDER BY q;
""")

,q,subs,mn,mx,churn
0,1,5500,0,31,58.56
1,2,5500,31,57,24.45
2,3,5500,57,86,9.49
3,4,5500,86,180,2.51


### E8 — SMS per active day and subscribers who never sent SMS
Calculate average SMS sent per active day, count subscribers who never sent an SMS, and show the population included.

In [129]:
q("""
SELECT ROUND(AVG(total_sms * 1.0 /
                 NULLIF(usage_days, 0)), 3) AS sms_per_active_day, -- SMS per active day
       SUM(CASE WHEN total_sms = 0 THEN 1 ELSE 0 END) AS never_sent_one, -- zero-SMS subscribers
       COUNT(*) AS subs                           -- population included
FROM customer_features
WHERE usage_days > 0;
""")

,sms_per_active_day,never_sent_one,subs
0,0.598,654,21996


# AREA F — ROAMING

### F1 — Roaming and non-roaming days and subscribers
From usage_daily, count roaming and non-roaming days, calculate their percentages, and count distinct subscribers in each group.

In [130]:
q("""
SELECT is_roaming,                               -- 0 non-roaming, 1 roaming
       COUNT(*) AS days,                         -- number of subscriber-days
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM usage_daily), 2) AS pct, -- share of usage days
       COUNT(DISTINCT customer_id) AS subs       -- distinct subscribers
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,days,pct,subs
0,0,1287143,97.02,21994
1,1,39596,2.98,16233


### F2 — Average data and voice on roaming versus non-roaming days
Compare average data_mb and voice_minutes on roaming and non-roaming days.

In [131]:
q("""
SELECT is_roaming,                               -- roaming status
       ROUND(AVG(data_mb), 2) AS avg_mb,         -- average data use
       ROUND(AVG(voice_minutes), 2) AS avg_voice -- average voice minutes
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,avg_mb,avg_voice
0,0,1572.95,56.27
1,1,1570.05,56.09


### F3 — Roaming by prepaid and postpaid subscribers
From customer_features, compare average roaming_days and the percentage who ever roamed for prepaid and postpaid subscribers.

In [132]:
q("""
SELECT connection_type,                          -- prepaid or postpaid
       COUNT(*) AS subs,                          -- subscriber count
       ROUND(AVG(roaming_days), 2) AS avg_roam_days, -- average roaming days
       ROUND(100.0 * AVG(CASE WHEN roaming_days > 0 THEN 1.0 ELSE 0.0 END), 1) AS pct_ever -- ever-roamed share
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type;
""")

,connection_type,subs,avg_roam_days,pct_ever
0,Postpaid,4851,1.84,74.8
1,Prepaid,17145,1.79,73.5


### F4 — Churn by roaming-day bands
 Create four roaming-day bands (0, 1–2, 3–5 and 6+). Calculate subscriber count and churn percentage for each band. Explain why the pattern may be misleading.

In [133]:
q("""
SELECT CASE                                     -- assign roaming-day band
         WHEN roaming_days = 0 THEN '0'
         WHEN roaming_days <= 2 THEN '1-2'
         WHEN roaming_days <= 5 THEN '3-5'
         ELSE '6+'
       END AS band,
       COUNT(*) AS subs,                         -- subscribers in band
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM customer_features
WHERE usage_days > 0
GROUP BY band
ORDER BY CASE band WHEN '0' THEN 1 WHEN '1-2' THEN 2 WHEN '3-5' THEN 3 ELSE 4 END;
""")

,band,subs,churn
0,0,5763,44.61
1,1-2,9922,21.58
2,3-5,5546,8.69
3,6+,765,3.92


### F5 — Five circles with highest roaming share
For each circle, calculate roaming days as a percentage of total usage days. Show the five highest circles.

In [134]:
q("""
SELECT circle_name,                              -- circle name
       ROUND(100.0 * SUM(roaming_days) /
             NULLIF(SUM(usage_days), 0), 2) AS pct_roaming -- roaming share of usage days
FROM customer_features
WHERE usage_days > 0
GROUP BY circle_name
ORDER BY pct_roaming DESC
LIMIT 5;
""")

,circle_name,pct_roaming
0,Odisha,3.07
1,Haryana,3.07
2,Delhi,3.07
3,Karnataka,3.05
4,Assam,3.05


# Airtel SQL Data Analysis — Areas G to N

# AREA G — COMPLAINTS

### G1 — Most common complaint category
What do subscribers complain about most? Show tickets and share by category.

In [135]:
q("""
SELECT category,
       COUNT(*) AS tickets,                                              -- tickets per category
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints), 1) AS pct -- share of all tickets
FROM complaints                                                          -- one row per ticket
GROUP BY category
ORDER BY tickets DESC;
""")

,category,tickets,pct
0,Slow Data,8035,29.9
1,Dropped Call,6365,23.7
2,Network Coverage,4885,18.2
3,Billing,3738,13.9
4,Recharge Issue,2679,10.0
5,SIM/KYC,1139,4.2


### G2 — Resolution by complaint channel
Does the complaint channel change how often a ticket is resolved?

In [136]:
q("""
SELECT channel,
       COUNT(*) AS tickets,                                              -- tickets per channel
       ROUND(100.0 * SUM(is_resolved) / COUNT(*), 2) AS pct_resolved     -- resolved share of all tickets
FROM complaints
GROUP BY channel
ORDER BY tickets DESC;
""")

,channel,tickets,pct_resolved
0,Store,6799,70.33
1,Call Center,6728,69.72
2,Twitter/X,6698,70.36
3,Airtel App,6616,70.04


### G3 — Open tickets
How many tickets are still open (is_resolved = 0)?

In [137]:
q("""
SELECT is_resolved,                                                      -- 0 open, 1 resolved
       COUNT(*) AS tickets,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints), 2) AS pct
FROM complaints
GROUP BY is_resolved;
""")

,is_resolved,tickets,pct
0,0,8021,29.88
1,1,18820,70.12


### G4 — Least resolved and slowest category
Which complaint category is resolved least often, and most slowly?

In [138]:
q("""
SELECT category,
       COUNT(*) AS tickets,
       ROUND(100.0 * SUM(is_resolved) / COUNT(*), 2) AS pct_resolved,    -- over all tickets
       ROUND(AVG(resolution_days), 2) AS avg_days                        -- closed tickets only
FROM complaints
GROUP BY category
ORDER BY pct_resolved;                                                   -- worst first
""")

,category,tickets,pct_resolved,avg_days
0,SIM/KYC,1139,68.57,7.65
1,Network Coverage,4885,69.52,7.60
2,Dropped Call,6365,69.71,7.46
3,Slow Data,8035,69.97,7.55
4,Billing,3738,70.87,7.50
5,Recharge Issue,2679,72.23,7.55


### G5 — Days to close a resolved ticket
How many days does a resolved ticket take to close? State the population.

In [139]:
q("""
SELECT ROUND(AVG(resolution_days), 2) AS avg_days,
       MIN(resolution_days) AS mn,                                       -- fastest
       MAX(resolution_days) AS mx,                                       -- slowest
       COUNT(*) AS closed                                                -- population: resolved tickets
FROM complaints
WHERE is_resolved = 1;
""")

,avg_days,mn,mx,closed
0,7.54,1.0,14.0,18820


### G6 — Never-complained vs complained churn
Do subscribers who never complain leave less often than those who do?

In [140]:
q("""
SELECT SUM(complaint_count = 0) AS never_,
       SUM(complaint_count > 0) AS did,
       ROUND(100.0 * AVG(CASE WHEN complaint_count = 0 THEN is_churned END), 2) AS churn_never,
       ROUND(100.0 * AVG(CASE WHEN complaint_count > 0 THEN is_churned END), 2) AS churn_did
FROM customer_features;
""")

,never_,did,churn_never,churn_did
0,6631,15369,20.03,25.36


### G7 — Complaint-count distribution
How many subscribers have raised 0, 1, 2 or more complaints?

In [141]:
q("""
SELECT complaint_count,
       COUNT(*) AS subs
FROM customer_features
GROUP BY complaint_count
ORDER BY complaint_count;
""")

,complaint_count,subs
0,0,6631
1,1,7854
2,2,4726
3,3,1926
4,4,625
5,5,183
6,6,44
7,7,10
8,8,1


### G8 — Circles with most complaints per subscriber
Which five circles raise the most complaints per subscriber?

In [142]:
q("""
SELECT circle_name,
       COUNT(*) AS subs,                                                 -- group size
       ROUND(1.0 * SUM(complaint_count) / COUNT(*), 3) AS per_sub        -- tickets per subscriber
FROM customer_features
GROUP BY circle_name
ORDER BY per_sub DESC
LIMIT 5;
""")

,circle_name,subs,per_sub
0,Tamil Nadu,731,1.291
1,Bihar,887,1.287
2,Delhi,735,1.275
3,Punjab,558,1.274
4,Kolkata,572,1.269


# AREA H — TENURE AND COHORTS

### H1 — Joiners per year
How many of our current subscribers joined in each year?

In [143]:
q("""
SELECT strftime('%Y', activation_date) AS yr,
       COUNT(*) AS subs
FROM customer_features
GROUP BY yr
ORDER BY yr;
""")

,yr,subs
0,2022,6922
1,2023,6925
2,2024,7053
3,2025,1100


### H2 — Churn by joining year
What share of each joining year has left?

In [144]:
q("""
SELECT strftime('%Y', activation_date) AS yr,
       COUNT(*) AS subs,
       SUM(is_churned) AS ch,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY yr
ORDER BY yr;
""")

,yr,subs,ch,churn
0,2022,6922,1374,19.85
1,2023,6925,1632,23.57
2,2024,7053,1901,26.95
3,2025,1100,319,29.00


### H3 — Churn by completed tenure year
How does churn change with each completed year on the network?

In [145]:
q("""
SELECT tenure_years,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,churn
0,0,4610,28.81
1,1,6961,24.78
2,2,7055,21.84
3,3,3374,18.73


### H4 — Time to churn
How long had a leaver been with us when they left?

In [146]:
q("""
SELECT ROUND(AVG(days_to_churn), 1) AS avg_days,
       MIN(days_to_churn) AS mn,
       MAX(days_to_churn) AS mx,
       COUNT(*) AS leavers
FROM customer_features
WHERE is_churned = 1;
""")

,avg_days,mn,mx,leavers
0,567.2,10,1259,5226


### H5 — Newest monthly cohorts
Are the newest monthly joining groups (last eight months) leaving faster?

In [147]:
q("""
SELECT strftime('%Y-%m', activation_date) AS cohort,
       COUNT(*) AS joined_,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
WHERE activation_date >= '2024-07-01'
GROUP BY cohort
ORDER BY cohort;
""")

,cohort,joined_,churn
0,2024-07,589,26.99
1,2024-08,593,31.03
2,2024-09,578,27.85
3,2024-10,568,28.70
4,2024-11,561,30.12
5,2024-12,621,27.86
6,2025-01,580,27.24
7,2025-02,520,30.96


### H6 — Observation window by tenure
Was every tenure group observed for the same number of days?

In [148]:
q("""
SELECT tenure_years,
       ROUND(AVG(observed_days), 1) AS avg_observed
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,avg_observed
0,0,149.2
1,1,158.8
2,2,161.6
3,3,164.2


### H7 — Survival by tenure
What share of each tenure group is still with us?

In [149]:
q("""
SELECT tenure_years,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(1 - is_churned), 2) AS surviving
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,surviving
0,0,4610,71.19
1,1,6961,75.22
2,2,7055,78.16
3,3,3374,81.27


### H8 — Spend by tenure
Do longer-standing subscribers pay more? Compare actual ARPU and six-month spend.

In [150]:
q("""
SELECT tenure_years,
       ROUND(AVG(actual_arpu_6m), 2) AS arpu,
       ROUND(AVG(recharge_total_6m), 0) AS spend_6m
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,arpu,spend_6m
0,0,329.10,1975.0
1,1,342.72,2056.0
2,2,350.13,2101.0
3,3,357.22,2143.0


# AREA I — CHURN: WHO LEAVES

### I1 — Overall churn
How many subscribers have we lost, and what is the churn rate?

In [151]:
q("""
SELECT COUNT(*) AS subs,
       SUM(is_churned) AS left_,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features;
""")

,subs,left_,churn
0,22000,5226,23.75


### I2 — Churn by connection type
What is the churn rate for prepaid and for postpaid subscribers?

In [152]:
q("""
SELECT connection_type,
       COUNT(*) AS subs,
       SUM(is_churned) AS ch,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subs,ch,churn
0,Postpaid,4851,785,16.18
1,Prepaid,17149,4441,25.90


### I3 — Six highest-churn plans
Which six plans have the highest churn rate?

In [153]:
q("""
SELECT plan_name, plan_type,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY plan_name, plan_type
ORDER BY churn DESC
LIMIT 6;
""")

,plan_name,plan_type,subs,churn
0,Prepaid Lite 99,Prepaid,996,31.12
1,Prepaid Smart 155,Prepaid,1137,30.17
2,Prepaid Entertainment 399,Prepaid,2142,26.24
3,Prepaid Quarterly 719,Prepaid,2561,25.85
4,Prepaid Value 209,Prepaid,1437,25.26
5,Prepaid Annual 1799,Prepaid,2901,25.13


### I4 — Churn by circle
Which circles have the highest and the lowest churn rate? Show all 22.

In [154]:
q("""
SELECT circle_name, region,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn DESC;
""")

,circle_name,region,subs,churn
0,Himachal Pradesh,North,1247,33.44
1,Maharashtra,West,1428,32.91
2,Andhra Pradesh,South,1412,28.90
3,Mumbai,West,1395,28.10
4,Uttar Pradesh (East),North,554,27.26
5,Haryana,North,1041,25.46
6,Jammu & Kashmir,North,933,24.87
7,Gujarat,West,1210,24.63
8,Assam,East,1301,23.14
9,Uttar Pradesh (West),North,1081,22.94


### I5 — Churn by device type
Do smartphone users leave at a different rate from feature-phone users?

In [155]:
q("""
SELECT device_type,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY device_type;
""")

,device_type,subs,churn
0,Feature Phone,3029,23.74
1,Smartphone,18971,23.76


### I6 — Churn by acquisition channel
Does churn differ by acquisition channel?

In [156]:
q("""
SELECT acquisition_channel,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY acquisition_channel
ORDER BY churn DESC;
""")

,acquisition_channel,subs,churn
0,Call Center,1770,24.63
1,Airtel App,6513,24.20
2,Website,3081,24.12
3,Doorstep,1349,23.80
4,Retailer,9287,23.15


### I7 — Churn by age band
Does churn differ by age band?

In [157]:
q("""
SELECT age_band,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY age_band
ORDER BY age_band;
""")

,age_band,subs,churn
0,18-24,5268,23.65
1,25-34,7467,23.70
2,35-44,4824,23.71
3,45-54,2895,23.49
4,55+,1546,24.97


### I8 — Attribute that separates leavers most
Which attribute separates leavers from stayers the most? Spread = highest group churn minus lowest group churn.

In [158]:
q("""
SELECT 'circle' AS k, ROUND(MAX(x) - MIN(x), 2) AS spread
FROM (SELECT 100.0 * AVG(is_churned) AS x FROM customer_features GROUP BY circle_name)
UNION ALL
SELECT 'plan' AS k, ROUND(MAX(x) - MIN(x), 2) AS spread
FROM (SELECT 100.0 * AVG(is_churned) AS x FROM customer_features GROUP BY plan_name)
UNION ALL
SELECT 'connection_type' AS k, ROUND(MAX(x) - MIN(x), 2) AS spread
FROM (SELECT 100.0 * AVG(is_churned) AS x FROM customer_features GROUP BY connection_type)
UNION ALL
SELECT 'device_type' AS k, ROUND(MAX(x) - MIN(x), 2) AS spread
FROM (SELECT 100.0 * AVG(is_churned) AS x FROM customer_features GROUP BY device_type)
UNION ALL
SELECT 'age_band' AS k, ROUND(MAX(x) - MIN(x), 2) AS spread
FROM (SELECT 100.0 * AVG(is_churned) AS x FROM customer_features GROUP BY age_band)
UNION ALL
SELECT 'acquisition_channel' AS k, ROUND(MAX(x) - MIN(x), 2) AS spread
FROM (SELECT 100.0 * AVG(is_churned) AS x FROM customer_features GROUP BY acquisition_channel)
ORDER BY spread DESC;
""")

,k,spread
0,circle,19.64
1,plan,15.48
2,connection_type,9.71
3,age_band,1.48
4,acquisition_channel,1.48
5,device_type,0.02


# AREA J — CHURN: WHY THEY LEAVE

### J1 — Open complaint and churn
Does an open complaint make a subscriber more likely to leave?

In [159]:
q("""
SELECT CASE WHEN complaint_count = 0 THEN '1 never complained'
            WHEN unresolved_count = 0 THEN '2 all resolved'
            ELSE '3 has open ticket' END AS grp,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY grp
ORDER BY grp;
""")

,grp,subs,churn
0,1 never complained,6631,20.03
1,2 all resolved,8689,23.75
2,3 has open ticket,6680,27.46


### J2 — Churn by open-ticket count
Does churn rise with each extra open ticket? Read the group sizes too.

In [160]:
q("""
SELECT unresolved_count,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY unresolved_count
ORDER BY unresolved_count;
""")

,unresolved_count,subs,churn
0,0,15320,22.14
1,1,5515,26.07
2,2,1018,33.10
3,3,120,37.50
4,4,25,48.00
5,5,2,100.00


### J3 — Churn by complaint category
Does what a subscriber complained about change their churn rate? Count subscribers, not tickets.

In [161]:
q("""
SELECT x.category,
       COUNT(DISTINCT x.customer_id) AS subs,
       ROUND(100.0 * AVG(c.is_churned), 2) AS churn
FROM complaints x
JOIN customer_features c ON c.customer_id = x.customer_id
GROUP BY x.category
ORDER BY churn DESC;
""")

,category,subs,churn
0,Billing,3401,28.49
1,Network Coverage,4371,27.80
2,SIM/KYC,1115,27.74
3,Dropped Call,5478,27.32
4,Slow Data,6685,27.27
5,Recharge Issue,2528,27.06


### J4 — Resolution speed and churn
Do slower resolutions lead to more churn? Keep 'none closed' as its own band.

In [162]:
q("""
SELECT CASE WHEN avg_resolution_days IS NULL THEN 'none closed'
            WHEN avg_resolution_days <= 4 THEN '1-4 days'
            WHEN avg_resolution_days <= 8 THEN '5-8 days'
            ELSE '9-14 days' END AS band,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
WHERE complaint_count > 0
GROUP BY band
ORDER BY band;
""")

,band,subs,churn
0,1-4 days,2810,24.09
1,5-8 days,4495,26.63
2,9-14 days,5248,25.02
3,none closed,2816,25.25


### J5 — Leaver vs stayer usage
How did leavers' usage differ from stayers'?

In [163]:
q("""
SELECT is_churned,
       COUNT(*) AS subs,
       ROUND(AVG(usage_days), 1) AS active_days,
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day,
       ROUND(AVG(recency_days), 1) AS days_quiet
FROM customer_features
WHERE usage_days > 0
GROUP BY is_churned;
""")

,is_churned,subs,active_days,mb_per_day,days_quiet
0,0,16772,69.9,1495.0,2.7
1,1,5224,29.7,1113.0,87.8


### J6 — Leaver vs stayer payments
How did leavers' payments differ from stayers'?

In [164]:
q("""
SELECT is_churned,
       COUNT(*) AS subs,
       ROUND(AVG(recharge_count), 2) AS payments,
       ROUND(AVG(actual_arpu_6m), 2) AS arpu
FROM customer_features
GROUP BY is_churned;
""")

,is_churned,subs,payments,arpu
0,0,16774,4.36,403.59
1,1,5226,2.02,154.68


### J7 — Monthly data use: leavers vs stayers
How did leavers' monthly data use compare with stayers', month by month?

In [165]:
q("""
SELECT m.month,
       ROUND(AVG(CASE WHEN c.is_churned = 1 THEN m.total_mb / 1024.0 END), 2) AS gb_leavers,
       ROUND(AVG(CASE WHEN c.is_churned = 0 THEN m.total_mb / 1024.0 END), 2) AS gb_stayers
FROM v_usage_month m
JOIN customer_features c ON c.customer_id = m.customer_id
GROUP BY m.month
ORDER BY m.month;
""")

,month,gb_leavers,gb_stayers
0,2025-01,11.30,17.77
1,2025-02,9.63,16.32
2,2025-03,10.29,18.16
3,2025-04,9.84,17.51
4,2025-05,10.91,21.20
5,2025-06,6.05,20.59


### J8 — Which driver moves churn most
Of spend, tenure, connection type and open tickets, which moves churn most? Gap = churn of one group minus churn of the other.

In [166]:
q("""
WITH c AS (
    SELECT is_churned, tenure_years, connection_type, unresolved_count,
           NTILE(4) OVER (ORDER BY recharge_total_6m, customer_id) AS m_score  -- 1 = lowest spend, 4 = highest
    FROM customer_features
)
SELECT * FROM (
    SELECT 'lowest vs highest spend' AS k,
           ROUND(100.0 * AVG(CASE WHEN m_score = 1 THEN is_churned END), 2) AS churn_a,
           ROUND(100.0 * AVG(CASE WHEN m_score = 4 THEN is_churned END), 2) AS churn_b,
           ROUND(100.0 * (AVG(CASE WHEN m_score = 1 THEN is_churned END)
                        - AVG(CASE WHEN m_score = 4 THEN is_churned END)), 2) AS gap
    FROM c
    UNION ALL
    SELECT 'tenure 0 yr vs 3 yr',
           ROUND(100.0 * AVG(CASE WHEN tenure_years = 0 THEN is_churned END), 2),
           ROUND(100.0 * AVG(CASE WHEN tenure_years = 3 THEN is_churned END), 2),
           ROUND(100.0 * (AVG(CASE WHEN tenure_years = 0 THEN is_churned END)
                        - AVG(CASE WHEN tenure_years = 3 THEN is_churned END)), 2)
    FROM c
    UNION ALL
    SELECT 'prepaid vs postpaid',
           ROUND(100.0 * AVG(CASE WHEN connection_type = 'Prepaid' THEN is_churned END), 2),
           ROUND(100.0 * AVG(CASE WHEN connection_type = 'Postpaid' THEN is_churned END), 2),
           ROUND(100.0 * (AVG(CASE WHEN connection_type = 'Prepaid' THEN is_churned END)
                        - AVG(CASE WHEN connection_type = 'Postpaid' THEN is_churned END)), 2)
    FROM c
    UNION ALL
    SELECT 'open ticket vs none',
           ROUND(100.0 * AVG(CASE WHEN unresolved_count > 0 THEN is_churned END), 2),
           ROUND(100.0 * AVG(CASE WHEN unresolved_count = 0 THEN is_churned END), 2),
           ROUND(100.0 * (AVG(CASE WHEN unresolved_count > 0 THEN is_churned END)
                        - AVG(CASE WHEN unresolved_count = 0 THEN is_churned END)), 2)
    FROM c
)
ORDER BY ABS(gap) DESC;
""")

,k,churn_a,churn_b,gap
0,lowest vs highest spend,54.82,9.00,45.82
1,tenure 0 yr vs 3 yr,28.81,18.73,10.08
2,prepaid vs postpaid,25.90,16.18,9.71
3,open ticket vs none,27.46,22.14,5.31


# AREA K — RECENCY, FREQUENCY AND MONETARY

### K1 — Churn by recency quartile
What is the churn rate in each recency quartile (r_score)? Note this is partly circular — recency reflects churn.

In [168]:
q("""
WITH c AS (
    SELECT is_churned,
           NTILE(4) OVER (ORDER BY recency_days DESC, customer_id) AS r_score  -- 4 = most recently active
    FROM customer_features
)
SELECT r_score,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM c
GROUP BY r_score
ORDER BY r_score;
""")

,r_score,subs,churn
0,1,5500,91.45
1,2,5500,3.53
2,3,5500,0.00
3,4,5500,0.04


### K2 — Churn by frequency quartile
What is the churn rate in each frequency quartile (f_score)?

In [172]:
q("""
WITH scored AS (
    SELECT recharge_count,
           is_churned,
           NTILE(4) OVER (ORDER BY recharge_count) AS f_score
    FROM customer_features
)
SELECT f_score,
       COUNT(*) AS subs,
       ROUND(AVG(recharge_count), 2) AS payments,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM scored
GROUP BY f_score
ORDER BY f_score;
""")

,f_score,subs,payments,churn
0,1,5500,0.33,43.09
1,2,5500,2.51,39.87
2,3,5500,5.46,8.33
3,4,5500,6.92,3.73


### K3 — Spend and churn by monetary quartile
What are the spend and the churn rate in each spend quartile (m_score)?

In [173]:
q("""
WITH scored AS (
    SELECT recharge_total_6m,
           is_churned,
           NTILE(4) OVER (ORDER BY recharge_total_6m) AS m_score
    FROM customer_features
)
SELECT m_score,
       COUNT(*) AS subs,
       ROUND(AVG(recharge_total_6m), 0) AS spend,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM scored
GROUP BY m_score
ORDER BY m_score;
""")

,m_score,subs,spend,churn
0,1,5500,167.0,54.82
1,2,5500,945.0,19.96
2,3,5500,1678.0,11.20
3,4,5500,5477.0,9.04


### K5 — Four payment segments
How big is each of four payment segments (from F and M scores), and how often does each leave?

In [174]:
q("""
WITH scored AS (
    SELECT *,
           NTILE(4) OVER (ORDER BY recharge_count) AS f_score,
           NTILE(4) OVER (ORDER BY recharge_total_6m) AS m_score
    FROM customer_features
)
SELECT CASE
         WHEN f_score >= 3 AND m_score >= 3 THEN 'Best'
         WHEN f_score >= 3 AND m_score < 3 THEN 'Frequent low value'
         WHEN f_score < 3 AND m_score >= 3 THEN 'Big but rare'
         ELSE 'At risk'
       END AS seg,
       COUNT(*) AS subs,
       ROUND(AVG(recharge_total_6m), 0) AS spend,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM scored
GROUP BY seg
ORDER BY churn;
""")

,seg,subs,spend,churn
0,Best,9189,3867.0,4.05
1,Frequent low value,1811,1062.0,12.37
2,Big but rare,1811,2108.0,40.92
3,At risk,9189,456.0,42.32


### K6 — Revenue share by segment
What share of monthly revenue does each segment carry?

In [176]:
q("""
WITH scored AS (
    SELECT *,
           NTILE(4) OVER (ORDER BY recharge_count) AS f_score,
           NTILE(4) OVER (ORDER BY recharge_total_6m) AS m_score
    FROM customer_features
),
segments AS (
    SELECT CASE
             WHEN f_score >= 3 AND m_score >= 3 THEN 'Best'
             WHEN f_score >= 3 THEN 'Frequent low value'
             WHEN m_score >= 3 THEN 'Big but rare'
             ELSE 'At risk'
           END AS seg,
           actual_arpu_6m
    FROM scored
)
SELECT seg,
       COUNT(*) AS subs,
       ROUND(SUM(actual_arpu_6m), 0) AS monthly_rev,
       ROUND(100.0 * SUM(actual_arpu_6m) /
             (SELECT SUM(actual_arpu_6m) FROM customer_features), 1) AS pct
FROM segments
GROUP BY seg;
""")

,seg,subs,monthly_rev,pct
0,At risk,9189,698926.0,9.2
1,Best,9189,5922512.0,78.2
2,Big but rare,1811,636343.0,8.4
3,Frequent low value,1811,320429.0,4.2


### K7 — Revenue in the At-risk segment
How much monthly revenue sits in the At risk segment?

In [178]:
q("""
WITH scored AS (
    SELECT *,
           NTILE(4) OVER (ORDER BY recharge_count) AS f_score,
           NTILE(4) OVER (ORDER BY recharge_total_6m) AS m_score
    FROM customer_features
)
SELECT COUNT(*) AS subs,
       ROUND(SUM(actual_arpu_6m), 0) AS monthly_rev,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM scored
WHERE f_score < 3 AND m_score < 3;
""")

,subs,monthly_rev,churn
0,9189,698926.0,42.32


### K8 — Circles with most Best-segment subscribers
Which five circles have the highest share of Best-segment subscribers?

In [180]:
q("""
WITH scored AS (
    SELECT *,
           NTILE(4) OVER (ORDER BY recharge_count) AS f_score,
           NTILE(4) OVER (ORDER BY recharge_total_6m) AS m_score
    FROM customer_features
)
SELECT circle_name,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(f_score >= 3 AND m_score >= 3), 1) AS pct_best
FROM scored
GROUP BY circle_name
ORDER BY pct_best DESC
LIMIT 5;
""")

,circle_name,subs,pct_best
0,Tamil Nadu,731,46.4
1,Delhi,735,46.1
2,Punjab,558,45.3
3,Rajasthan,1082,45.2
4,Uttar Pradesh (West),1081,44.9


# AREA L — CIRCLE AND REGION

### L1 — Region churn and revenue
What are the churn rate and the monthly revenue of each region?

In [181]:
q("""
SELECT region,
       COUNT(*) AS subs,
       ROUND(SUM(actual_arpu_6m), 0) AS monthly_rev,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY region
ORDER BY churn DESC;
""")

,region,subs,monthly_rev,churn
0,West,4665,1571780.0,27.85
1,North,7231,2477493.0,23.73
2,East,5459,1893200.0,21.91
3,South,4645,1635737.0,21.85


### L2 — Four highest-churn circles
Which four circles churn the most, and what do their subscribers pay?

In [182]:
q("""
SELECT circle_name, region,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn,
       ROUND(AVG(actual_arpu_6m), 2) AS arpu
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn DESC
LIMIT 4;
""")

,circle_name,region,subs,churn,arpu
0,Himachal Pradesh,North,1247,33.44,313.16
1,Maharashtra,West,1428,32.91,315.15
2,Andhra Pradesh,South,1412,28.90,328.17
3,Mumbai,West,1395,28.10,339.27


### L3 — Four lowest-churn circles
Which four circles churn the least?

In [183]:
q("""
SELECT circle_name, region,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn
LIMIT 4;
""")

,circle_name,region,subs,churn
0,Punjab,North,558,13.80
1,Delhi,North,735,16.19
2,Karnataka,South,1361,17.78
3,West Bengal,East,685,19.12


### L5 — Circle gap among prepaid only
Among prepaid subscribers only, is the gap between the best and worst circles still there? (Replace the circle names below with your own L2/L3 results if they differ.)

In [184]:
q("""
SELECT circle_name,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
WHERE connection_type = 'Prepaid'
  AND circle_name IN ('Himachal Pradesh', 'Maharashtra', 'Punjab', 'Delhi')
GROUP BY circle_name
ORDER BY churn DESC;
""")

,circle_name,subs,churn
0,Himachal Pradesh,1005,35.92
1,Maharashtra,1125,35.56
2,Delhi,571,17.51
3,Punjab,442,14.93


### L6 — Circle sizes
How many subscribers do the smallest, the largest and the average circle hold?

In [185]:
q("""
SELECT MIN(n) AS smallest,
       MAX(n) AS largest,
       ROUND(AVG(n), 0) AS average_
FROM (SELECT COUNT(*) AS n FROM customer_features GROUP BY circle_name);
""")

,smallest,largest,average_
0,554,1428,1000.0


### L7 — Regional ARPU and data use
Do regions differ in revenue per subscriber and in data use?

In [186]:
q("""
SELECT region,
       COUNT(*) AS subs,
       ROUND(AVG(actual_arpu_6m), 2) AS arpu,
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day
FROM customer_features
GROUP BY region
ORDER BY arpu DESC;
""")

,region,subs,arpu,mb_per_day
0,South,4645,352.15,1430.0
1,East,5459,346.80,1404.0
2,North,7231,342.62,1389.0
3,West,4665,336.93,1403.0


# AREA M — REVENUE CONCENTRATION

### M1 — Top 1, 10, 100, 1,000 spenders
What share of total spend do our top 1, 10, 100 and 1,000 subscribers bring?

In [187]:
q("""
WITH o AS (
    SELECT customer_id,
           recharge_total_6m AS rev,
           ROW_NUMBER() OVER (ORDER BY recharge_total_6m DESC, customer_id) AS rn,
           SUM(recharge_total_6m) OVER (ORDER BY recharge_total_6m DESC, customer_id) AS cum
    FROM customer_features
)
SELECT rn, rev, cum,
       ROUND(100.0 * cum / (SELECT SUM(recharge_total_6m) FROM customer_features), 2) AS pct
FROM o
WHERE rn IN (1, 10, 100, 1000)
ORDER BY rn;
""")

,rn,rev,cum,pct
0,1,10939,10939,0.02
1,10,10741,107964,0.24
2,100,10478,1060776,2.33
3,1000,9695,10158989,22.34


### M2 — Spend by decile
What share of total spend does each tenth of the base bring?

In [188]:
q("""
WITH d AS (
    SELECT recharge_total_6m AS rev,
           NTILE(10) OVER (ORDER BY recharge_total_6m DESC) AS dec_
    FROM customer_features
),
g AS (SELECT dec_, SUM(rev) AS rev FROM d GROUP BY dec_)
SELECT dec_, rev,
       ROUND(100.0 * rev / (SELECT SUM(rev) FROM g), 2) AS pct,
       ROUND(100.0 * SUM(rev) OVER (ORDER BY dec_) / (SELECT SUM(rev) FROM g), 2) AS cum_pct
FROM g
ORDER BY dec_;
""")

,dec_,rev,pct,cum_pct
0,1,18699379,41.13,41.13
1,2,8848274,19.46,60.59
2,3,4824531,10.61,71.20
3,4,3914201,8.61,79.80
4,5,3066739,6.74,86.55
5,6,2410893,5.30,91.85
6,7,1997121,4.39,96.24
7,8,1488446,3.27,99.52
8,9,219684,0.48,100.00
9,10,0,0.00,100.00


### M3 — Share of base bringing 80% of spend
What share of subscribers brings in 80% of total spend?

In [189]:
q("""
WITH o AS (
    SELECT recharge_total_6m AS rev,
           ROW_NUMBER() OVER (ORDER BY recharge_total_6m DESC, customer_id) AS rn,
           SUM(recharge_total_6m) OVER (ORDER BY recharge_total_6m DESC, customer_id) AS cum
    FROM customer_features
),
t AS (SELECT SUM(recharge_total_6m) AS s FROM customer_features)
SELECT MIN(rn) AS subs_needed,
       ROUND(100.0 * MIN(rn) / (SELECT COUNT(*) FROM customer_features), 1) AS pct_of_base
FROM o, t
WHERE 100.0 * cum / s >= 80;
""")

,subs_needed,pct_of_base
0,8860,40.3


### M4 — Circles dependent on top spenders
Which five circles depend most on their top tenth of spenders?

In [190]:
q("""
WITH d AS (
    SELECT circle_name,
           recharge_total_6m AS rev,
           NTILE(10) OVER (PARTITION BY circle_name ORDER BY recharge_total_6m DESC) AS dec_
    FROM customer_features
)
SELECT circle_name,
       ROUND(100.0 * SUM(CASE WHEN dec_ = 1 THEN rev END) / SUM(rev), 1) AS top_decile_pct
FROM d
GROUP BY circle_name
ORDER BY top_decile_pct DESC
LIMIT 5;
""")

,circle_name,top_decile_pct
0,Himachal Pradesh,43.5
1,Odisha,42.6
2,Andhra Pradesh,42.6
3,Mumbai,42.4
4,Jammu & Kashmir,41.7


### M5 — Revenue from leavers
What share of monthly revenue came from subscribers who have since left?

In [191]:
q("""
SELECT is_churned,
       COUNT(*) AS subs,
       ROUND(SUM(actual_arpu_6m), 0) AS monthly_rev,
       ROUND(100.0 * SUM(actual_arpu_6m) / (SELECT SUM(actual_arpu_6m) FROM customer_features), 2) AS pct
FROM customer_features
GROUP BY is_churned;
""")

,is_churned,subs,monthly_rev,pct
0,0,16774,6769858.0,89.33
1,1,5226,808353.0,10.67


### M6 — Data vs spend concentration
Is data use concentrated in the top tenth as much as spend is?

In [192]:
q("""
WITH d AS (
    SELECT total_data_mb AS mb,
           recharge_total_6m AS rev,
           NTILE(10) OVER (ORDER BY total_data_mb DESC) AS mdec,
           NTILE(10) OVER (ORDER BY recharge_total_6m DESC) AS rdec
    FROM customer_features
)
SELECT ROUND(100.0 * SUM(CASE WHEN mdec = 1 THEN mb END) / SUM(mb), 2) AS top_decile_of_data,
       ROUND(100.0 * SUM(CASE WHEN rdec = 1 THEN rev END) / SUM(rev), 2) AS top_decile_of_revenue
FROM d;
""")

,top_decile_of_data,top_decile_of_revenue
0,42.43,41.13


# AREA N — A/B EXPERIMENT

### N1 — Group sizes
How many subscribers are in group A and in group B?

In [193]:
q("""
SELECT ab_group,
       COUNT(*) AS subs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 2) AS pct
FROM customer_features
GROUP BY ab_group;
""")

,ab_group,subs,pct
0,A,10996,49.98
1,B,11004,50.02


### N2 — Pre-offer balance
Did groups A and B look alike before the offer began?

In [194]:
q("""
SELECT ab_group,
       ROUND(AVG(stated_arpu), 2) AS stated_arpu,
       ROUND(100.0 * AVG(connection_type = 'Postpaid'), 2) AS pct_post,
       ROUND(AVG(tenure_days), 1) AS tenure,
       ROUND(100.0 * AVG(device_type = 'Smartphone'), 2) AS pct_smart,
       ROUND(AVG(complaint_count), 3) AS complaints
FROM customer_features
GROUP BY ab_group;
""")

,ab_group,stated_arpu,pct_post,tenure,pct_smart,complaints
0,A,424.30,22.02,697.7,85.64,1.228
1,B,418.99,22.08,697.5,86.82,1.212


### N3 — Monthly data use by group
How did daily data use change, month by month, in each group?

In [195]:
q("""
SELECT strftime('%Y-%m', u.usage_date) AS m,
       c.ab_group,
       ROUND(AVG(u.data_mb), 2) AS avg_mb
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
GROUP BY m, c.ab_group
ORDER BY m, c.ab_group;
""")

,m,ab_group,avg_mb
0,2025-01,A,1474.44
1,2025-01,B,1447.24
2,2025-02,A,1483.57
3,2025-02,B,1470.14
4,2025-03,A,1492.60
5,2025-03,B,1478.46
6,2025-04,A,1508.76
7,2025-04,B,1485.11
8,2025-05,A,1520.39
9,2025-05,B,2016.39


### N4 — Data per day, before vs during offer
How much more data per day did a group B subscriber use during the offer? Average per subscriber first, then average those.

In [196]:
q("""
WITH p AS (
    SELECT c.ab_group, u.customer_id,
           CASE WHEN u.usage_date >= '2025-05-01' THEN 'during' ELSE 'before' END AS per,
           AVG(u.data_mb) AS m
    FROM usage_daily u
    JOIN customer_features c ON c.customer_id = u.customer_id
    GROUP BY u.customer_id, per
)
SELECT per, ab_group,
       COUNT(*) AS subs,
       ROUND(AVG(m), 2) AS avg_mb_per_sub
FROM p
GROUP BY per, ab_group
ORDER BY per, ab_group;
""")

,per,ab_group,subs,avg_mb_per_sub
0,before,A,10990,1342.76
1,before,B,11003,1333.38
2,during,A,9247,1393.33
3,during,B,9201,1866.70


### N5 — Churn by group
What is the churn rate in group A and in group B? (Descriptive only — the significance test belongs to Step 4.)

In [197]:
q("""
SELECT ab_group,
       COUNT(*) AS subs,
       SUM(is_churned) AS ch,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
GROUP BY ab_group;
""")

,ab_group,subs,ch,churn
0,A,10996,2587,23.53
1,B,11004,2639,23.98


### N6 — ARPU by group
Does revenue per subscriber differ between the groups, on either ARPU column?

In [198]:
q("""
SELECT ab_group,
       ROUND(AVG(stated_arpu), 2) AS stated,
       ROUND(AVG(actual_arpu_6m), 2) AS actual
FROM customer_features
GROUP BY ab_group;
""")

,ab_group,stated,actual
0,A,424.30,347.58
1,B,418.99,341.35


### N7 — Payments during the offer
How much did a subscriber in each group pay during May–June? Non-payers count as zero.

In [199]:
q("""
SELECT c.ab_group,
       COUNT(*) AS subs,
       ROUND(AVG(COALESCE(w.paid, 0)), 2) AS avg_paid_may_jun
FROM customer_features c
LEFT JOIN (SELECT customer_id, SUM(amount_inr) AS paid
           FROM recharges
           WHERE recharge_date BETWEEN '2025-05-01' AND '2025-06-30'
           GROUP BY customer_id) w
       ON w.customer_id = c.customer_id
GROUP BY c.ab_group;
""")

,ab_group,subs,avg_paid_may_jun
0,A,10996,524.77
1,B,11004,517.82


### N8 — Allowance used during the offer
What share of their data allowance did each group use during the offer?

In [200]:
q("""
WITH d AS (
    SELECT c.ab_group, c.customer_id, c.monthly_gb,
           SUM(u.data_mb) / 1024.0 / 2 AS gb_per_month
    FROM usage_daily u
    JOIN customer_features c ON c.customer_id = u.customer_id
    WHERE u.usage_date >= '2025-05-01'
    GROUP BY c.customer_id
)
SELECT ab_group,
       COUNT(*) AS subs,
       ROUND(AVG(gb_per_month), 2) AS used_gb,
       ROUND(AVG(monthly_gb), 1) AS allowance_gb,
       ROUND(100.0 * AVG(gb_per_month / monthly_gb), 1) AS pct_used
FROM d
GROUP BY ab_group;
""")

,ab_group,subs,used_gb,allowance_gb,pct_used
0,A,9247,16.77,79.0,21.1
1,B,9201,22.22,77.9,28.5
